In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# CONFIGURATION
# ============================================================

CATALOG = "aml_engine"
SCHEMA = "aml_poc"

SILVER_TX_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"
SILVER_ACCOUNTS_TABLE = f"{CATALOG}.{SCHEMA}.silver_accounts"

RULE_RESULTS_TABLE = f"{CATALOG}.{SCHEMA}.rule_results"

S3_BASE_PATH = "s3://zubair-s3-demo/raw_dataset/aml"
S3_DELTA_PATH = f"{S3_BASE_PATH}/delta_tables"

RULE_RESULTS_PATH = f"{S3_DELTA_PATH}/rule_results"


# ============================================================
# RULE PARAMETERS
# ============================================================

# Make thresholds configurable.
# Do not hard-code them throughout the rule logic.

HIGH_VALUE_THRESHOLD = 1000.0

# Your TIMESTAMP is a simulated time-step (0, 1, 2, ...).
VELOCITY_TIME_WINDOW = 2

# Number of transactions within the time window
# considered unusual for this POC.
VELOCITY_THRESHOLD = 5

# Distinct accounts involved in fan-in/fan-out
FAN_IN_THRESHOLD = 5
FAN_OUT_THRESHOLD = 5


print("Rule Engine configuration loaded.")

In [0]:
# COMMAND ----------

transactions_df = spark.table(SILVER_TX_TABLE)

print(f"Silver transactions: {transactions_df.count()}")

display(transactions_df.limit(10))

In [0]:
# COMMAND ----------

accounts_df = spark.table(SILVER_ACCOUNTS_TABLE)

print(f"Silver accounts: {accounts_df.count()}")

display(accounts_df.limit(10))

tx_amount > HIGH_VALUE_THRESHOLD

In [0]:
# COMMAND ----------

rule_high_value = (
    transactions_df
    .select(
        "tx_id",
        "sender_account_id",
        "receiver_account_id",
        "tx_amount",
        "event_time"
    )
    .withColumn(
        "rule_id",
        F.lit("R001")
    )
    .withColumn(
        "rule_name",
        F.lit("HIGH_VALUE_TRANSACTION")
    )
    .withColumn(
        "rule_category",
        F.lit("AMOUNT")
    )
    .withColumn(
        "rule_triggered",
        F.when(
            F.col("tx_amount") > HIGH_VALUE_THRESHOLD,
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "rule_score",
        F.when(
            F.col("rule_triggered"),
            F.lit(30)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "rule_evidence",
        F.when(
            F.col("rule_triggered"),
            F.concat(
                F.lit("Transaction amount "),
                F.col("tx_amount").cast("string"),
                F.lit(" exceeds threshold "),
                F.lit(str(HIGH_VALUE_THRESHOLD))
            )
        ).otherwise(F.lit(None))
    )
)

same sender
+
many transactions
+
within a short time window
=
potentially suspicious velocity

In [0]:
# COMMAND ----------

velocity_window = (
    Window
    .partitionBy("sender_account_id")
    .orderBy("event_time")
    .rangeBetween(
        -VELOCITY_TIME_WINDOW,
        VELOCITY_TIME_WINDOW
    )
)

In [0]:
# COMMAND ----------

velocity_df = (
    transactions_df
    .withColumn(
        "velocity_count",
        F.count("tx_id").over(velocity_window)
    )
)

In [0]:
# COMMAND ----------

rule_velocity = (
    velocity_df
    .select(
        "tx_id",
        "sender_account_id",
        "receiver_account_id",
        "tx_amount",
        "event_time",
        "velocity_count"
    )
    .withColumn(
        "rule_id",
        F.lit("R002")
    )
    .withColumn(
        "rule_name",
        F.lit("TRANSACTION_VELOCITY")
    )
    .withColumn(
        "rule_category",
        F.lit("VELOCITY")
    )
    .withColumn(
        "rule_triggered",
        F.col("velocity_count") >= VELOCITY_THRESHOLD
    )
    .withColumn(
        "rule_score",
        F.when(
            F.col("rule_triggered"),
            F.lit(20)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "rule_evidence",
        F.when(
            F.col("rule_triggered"),
            F.concat(
                F.lit("Sender executed "),
                F.col("velocity_count").cast("string"),
                F.lit(" transactions within time window ")
            )
        ).otherwise(F.lit(None))
    )
)

Rule R003 — Fan-In

In [0]:
# COMMAND ----------

fan_in_df = (
    transactions_df
    .groupBy(
        "receiver_account_id",
        "event_time"
    )
    .agg(
        F.countDistinct("sender_account_id")
            .alias("unique_senders"),

        F.sum("tx_amount")
            .alias("total_inflow")
    )
)

In [0]:
# COMMAND ----------

rule_fan_in = (
    transactions_df
    .join(
        fan_in_df,
        on=[
            "receiver_account_id",
            "event_time"
        ],
        how="left"
    )
    .select(
        "tx_id",
        "sender_account_id",
        "receiver_account_id",
        "tx_amount",
        "event_time",
        "unique_senders",
        "total_inflow"
    )
    .withColumn(
        "rule_id",
        F.lit("R003")
    )
    .withColumn(
        "rule_name",
        F.lit("FAN_IN")
    )
    .withColumn(
        "rule_category",
        F.lit("NETWORK")
    )
    .withColumn(
        "rule_triggered",
        F.col("unique_senders") >= FAN_IN_THRESHOLD
    )
    .withColumn(
        "rule_score",
        F.when(
            F.col("rule_triggered"),
            F.lit(25)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "rule_evidence",
        F.when(
            F.col("rule_triggered"),
            F.concat(
                F.lit("Receiver received funds from "),
                F.col("unique_senders").cast("string"),
                F.lit(" distinct senders at event time ")
            )
        ).otherwise(F.lit(None))
    )
)

Rule R004 — Fan-Out

In [0]:
# COMMAND ----------

fan_out_df = (
    transactions_df
    .groupBy(
        "sender_account_id",
        "event_time"
    )
    .agg(
        F.countDistinct("receiver_account_id")
            .alias("unique_receivers"),

        F.sum("tx_amount")
            .alias("total_outflow")
    )
)

In [0]:
# COMMAND ----------

rule_fan_out = (
    transactions_df
    .join(
        fan_out_df,
        on=[
            "sender_account_id",
            "event_time"
        ],
        how="left"
    )
    .select(
        "tx_id",
        "sender_account_id",
        "receiver_account_id",
        "tx_amount",
        "event_time",
        "unique_receivers",
        "total_outflow"
    )
    .withColumn(
        "rule_id",
        F.lit("R004")
    )
    .withColumn(
        "rule_name",
        F.lit("FAN_OUT")
    )
    .withColumn(
        "rule_category",
        F.lit("NETWORK")
    )
    .withColumn(
        "rule_triggered",
        F.col("unique_receivers") >= FAN_OUT_THRESHOLD
    )
    .withColumn(
        "rule_score",
        F.when(
            F.col("rule_triggered"),
            F.lit(25)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "rule_evidence",
        F.when(
            F.col("rule_triggered"),
            F.concat(
                F.lit("Sender transferred funds to "),
                F.col("unique_receivers").cast("string"),
                F.lit(" distinct receivers at event time ")
            )
        ).otherwise(F.lit(None))
    )
)

In [0]:
# COMMAND ----------

rule_results_df = (
    rule_high_value
    .unionByName(
        rule_velocity,
        allowMissingColumns=True
    )
    .unionByName(
        rule_fan_in,
        allowMissingColumns=True
    )
    .unionByName(
        rule_fan_out,
        allowMissingColumns=True
    )
)

In [0]:
# COMMAND ----------

rule_results_df = (
    rule_results_df
    .withColumn(
        "rule_execution_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "rule_version",
        F.lit("v1.0")
    )
)

In [0]:
# COMMAND ----------

triggered_rule_results_df = (
    rule_results_df
    .filter(F.col("rule_triggered") == True)
)

In [0]:
# COMMAND ----------

(
    triggered_rule_results_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "path",
        RULE_RESULTS_PATH
    )
    .saveAsTable(RULE_RESULTS_TABLE)
)

print("Rule Results table created successfully.")
print(f"Unity Catalog : {RULE_RESULTS_TABLE}")
print(f"S3 Location   : {RULE_RESULTS_PATH}")

In [0]:
%sql
-- COMMAND ----------

SELECT *
FROM aml_engine.aml_poc.rule_results
ORDER BY tx_id
LIMIT 50;

In [0]:
transaction_rule_scores = (
    triggered_rule_results_df
    .groupBy("tx_id")
    .agg(
        F.sum("rule_score").alias("rule_score"),
        F.collect_set("rule_id").alias("triggered_rule_ids"),
        F.collect_set("rule_name").alias("triggered_rules")
    )
)

In [0]:
RULE_SCORE_TABLE = f"{CATALOG}.{SCHEMA}.rule_transaction_scores"

RULE_SCORE_PATH = (
    f"{S3_DELTA_PATH}/rule_transaction_scores"
)

(
    transaction_rule_scores.write
    .format("delta")
    .mode("overwrite")
    .option("path", RULE_SCORE_PATH)
    .saveAsTable(RULE_SCORE_TABLE)
)

In [0]:
%sql
SELECT *
FROM aml_engine.aml_poc.rule_transaction_scores
ORDER BY tx_id
